In [1]:
import pandas as pd
import numpy as np
import os

# 🔹 Path
folder_path = r"D:\Swapnil\Work\Projects\FIFA\Updated Base Files"

# 🔹 Load files
sales = pd.read_excel(os.path.join(folder_path, "Coco-Cola_Final.xlsx"))
countries = pd.read_excel(os.path.join(folder_path, "Dim_Country_Final.xlsx"))

print("Sales shape:", sales.shape)
print("Countries shape:", countries.shape)

# --------------------------------------------------
# 🔹 STEP 4.1 — Expand to 48 countries
# --------------------------------------------------

expanded_sales = sales.merge(countries, how="cross")

print("After country expansion:", expanded_sales.shape)

# --------------------------------------------------
# 🔹 STEP 4.2 — Add company layer
# --------------------------------------------------

companies = ["Coca-Cola", "Pepsi", "Red Bull"]

expanded_sales = pd.concat(
    [expanded_sales.assign(company_name=c) for c in companies],
    ignore_index=True
)

print("After company expansion:", expanded_sales.shape)

# --------------------------------------------------
# 🔹 STEP 4.3 — Create simulation columns
# --------------------------------------------------

expanded_sales["sim_units_sold"] = expanded_sales["units_sold"]
expanded_sales["sim_price_per_unit"] = expanded_sales["price_per_unit"]
expanded_sales["sim_total_sales"] = expanded_sales["total_sales"]
expanded_sales["sim_operating_profit"] = expanded_sales["operating_profit"]

# --------------------------------------------------
# 🔹 STEP 4.4 — Apply company behavior
# --------------------------------------------------

# Pepsi
mask_pepsi = expanded_sales["company_name"] == "Pepsi"
expanded_sales.loc[mask_pepsi, "sim_total_sales"] *= 0.95
expanded_sales.loc[mask_pepsi, "sim_operating_profit"] *= 0.92

# Red Bull
mask_rb = expanded_sales["company_name"] == "Red Bull"
expanded_sales.loc[mask_rb, "sim_units_sold"] *= 0.60
expanded_sales.loc[mask_rb, "sim_price_per_unit"] *= 1.40
expanded_sales.loc[mask_rb, "sim_total_sales"] *= 0.90
expanded_sales.loc[mask_rb, "sim_operating_profit"] *= 1.15

# --------------------------------------------------
# 🔹 STEP 4.5 — Apply country multiplier
# --------------------------------------------------

expanded_sales["sim_units_sold"] *= expanded_sales["country_multiplier"]
expanded_sales["sim_total_sales"] *= expanded_sales["country_multiplier"]
expanded_sales["sim_operating_profit"] *= expanded_sales["country_multiplier"]

# --------------------------------------------------
# 🔹 STEP 4.6 — Apply host boost
# --------------------------------------------------

mask_host = expanded_sales["is_host"] == 1

expanded_sales.loc[mask_host, "sim_total_sales"] *= 1.25
expanded_sales.loc[mask_host, "sim_operating_profit"] *= 1.20

# --------------------------------------------------
# 🔹 STEP 4.7 — Final check
# --------------------------------------------------

print("\nFINAL DATASET SHAPE:", expanded_sales.shape)

print("\nSample rows:")
print(expanded_sales[[
    "country",
    "company_name",
    "sim_total_sales"
]].head(10))

# --------------------------------------------------
# 🔹 STEP 4.8 — Save output
# --------------------------------------------------

output_path = os.path.join(folder_path, "Step4_Global_Expanded.xlsx")

expanded_sales.to_excel(output_path, index=False)

print("\nSaved successfully to:", output_path)

Sales shape: (9648, 13)
Countries shape: (48, 8)
After country expansion: (463104, 21)
After company expansion: (1389312, 22)

FINAL DATASET SHAPE: (1389312, 26)

Sample rows:
         country company_name  sim_total_sales
0         Canada    Coca-Cola           9000.0
1         Mexico    Coca-Cola           9375.0
2  United States    Coca-Cola           9750.0
3      Argentina    Coca-Cola           7500.0
4         Brazil    Coca-Cola           7800.0
5       Colombia    Coca-Cola           6300.0
6        Ecuador    Coca-Cola           6000.0
7       Paraguay    Coca-Cola           5100.0
8        Uruguay    Coca-Cola           6000.0
9        Austria    Coca-Cola           6000.0


ValueError: This sheet is too large! Your sheet size is: 1389312, 26 Max sheet size is: 1048576, 16384

In [2]:
output_path = os.path.join(folder_path, "Step4_Global_Expanded.csv")

expanded_sales.to_csv(output_path, index=False)

print("Saved as CSV:", output_path)

Saved as CSV: D:\Swapnil\Work\Projects\FIFA\Updated Base Files\Step4_Global_Expanded.csv
